# 🛍️ Fashion AI Chatbot
## Image-to-Caption + Visual Similarity Search

> **How it works:**
> 1. Upload a product image (local file or URL)
> 2. The EfficientNetB0 + Transformer model generates a natural-language caption
> 3. Cosine similarity retrieves the **Top-K most visually similar** products from the dataset
> 4. Results are displayed with similarity scores, metadata, and product links

---
### 📋 Sections
| # | Section |
|---|---------|
| 1 | Imports & Setup |
| 2 | Configuration |
| 3 | Load Dataset |
| 4 | Model Architecture |
| 5 | Load Model & Tokenizer |
| 6 | Image Utilities |
| 7 | Caption Generation |
| 8 | Embedding Extractor |
| 9 | Load Cached Embeddings |
| 10 | 💬 **Chatbot Interface** |


In [1]:
import os, re, json, warnings, textwrap
from pathlib import Path
from io import BytesIO
import ipywidgets as widgets
from IPython.display import display, clear_output, HTML
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import tensorflow as tf
from tensorflow import keras
from tensorflow.keras import layers
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input
from tensorflow.keras.applications import EfficientNetB0
from tensorflow.keras.preprocessing.text import tokenizer_from_json
from tensorflow.keras.preprocessing.sequence import pad_sequences
from PIL import Image
import requests
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
from sklearn.metrics.pairwise import cosine_similarity

try:
    import faiss
    FAISS_AVAILABLE = True
    print("✅ FAISS available")
except ImportError:
    FAISS_AVAILABLE = False
    print("ℹ️  FAISS not installed — using sklearn cosine similarity")

print(f"\n✅ TensorFlow {tf.__version__}")
print(f"✅ NumPy      {np.__version__}")
print(f"✅ Pandas     {pd.__version__}")
print(f"✅ Widgets    {widgets.__version__}")


✅ FAISS available

✅ TensorFlow 2.21.0
✅ NumPy      2.4.4
✅ Pandas     3.0.2
✅ Widgets    8.1.8


## Section 2 — Configuration

In [2]:
# ══════════════════════════════════════════════════════════════
# ⚙️  CONFIGURE THESE PATHS FOR YOUR ENVIRONMENT
# ══════════════════════════════════════════════════════════════

CSV_PATH        = "cleaned_data_final.csv"
IMG_DIR         = Path(r"E:\AMANY1\ChatBot AI Fashion Assistant\scraping\data\images_fresh")
MODELS_DIR      = Path("saved_models")
EMBEDDINGS_DIR  = Path("embeddings_cache")

# Saved artifacts
MODEL4_WEIGHTS   = MODELS_DIR / "model4_efficientnetb0_transformer_best.keras"
TOKENIZER_PATH   = MODELS_DIR / "tokenizer.json"
EMBEDDINGS_PATH  = EMBEDDINGS_DIR / "product_embeddings.npy"
PRODUCT_IDS_PATH = EMBEDDINGS_DIR / "product_ids.npy"

# Image settings (must match training)
IMG_SIZE      = (224, 224)
IMG_CHANNELS  = 3

# Model hyperparameters (must match training)
VOCAB_SIZE    = 4000
MAX_SEQ_LEN   = 32
EMBED_DIM     = 512
UNITS         = 512
ATTN_UNITS    = 256
NUM_HEADS     = 4
FF_DIM        = 512
DROPOUT_RATE  = 0.3

# Retrieval
TOP_K              = 5
EFFNET_FEATURE_DIM = 1280

print("✅ Configuration loaded")


✅ Configuration loaded


## Section 3 — Load Dataset

In [3]:
df = pd.read_csv(CSV_PATH, low_memory=False)
df = df[df["img_url_valid"] == True].reset_index(drop=True)
print(f"✅ Dataset loaded: {len(df):,} rows with valid images")

START_TOKEN = "<start>"
END_TOKEN   = "<end>"
UNK_TOKEN   = "<unk>"
PAD_TOKEN   = "<pad>"

def build_caption(row):
    title = str(row.get("title_clean", "")).strip()
    title = re.sub(r"[\u0600-\u06FF]+", "", title)
    title = re.sub(r"\s+", " ", title).strip()
    title = " ".join(title.split()[:15])
    parts = [title]
    for field, key in [("primary_color","primary_color"),("category_clean","category_clean"),
                        ("gender_clean","gender_clean"),("price_bucket","price_bucket")]:
        val = str(row.get(key, "")).strip().lower()
        if val and val not in ("unknown","nan",""):
            parts.append(val)
    price = str(row.get("price_egp","")).strip()
    if price and price != "nan":
        parts.append(f"egp {price}")
    caption = re.sub(r"\s+", " ", " ".join(parts)).strip()
    return f"{START_TOKEN} {caption} {END_TOKEN}"

df["caption"] = df.apply(build_caption, axis=1)
print(f"✅ Captions built for {len(df):,} products")


✅ Dataset loaded: 5,281 rows with valid images
✅ Captions built for 5,281 products


## Section 4 — Model Architecture (EfficientNetB0 + Transformer)

In [4]:
class TransformerDecoderBlock(layers.Layer):
    def __init__(self, embed_dim, num_heads, ff_dim, dropout=0.1, **kwargs):
        super().__init__(**kwargs)
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.ff_dim    = ff_dim
        key_dim = embed_dim // num_heads
        self.attn1 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, name="self_attn")
        self.attn2 = layers.MultiHeadAttention(num_heads=num_heads, key_dim=key_dim, name="cross_attn")
        self.ff1   = layers.Dense(ff_dim, activation="gelu", name="ff1")
        self.ff2   = layers.Dense(embed_dim, name="ff2")
        self.norm1 = layers.LayerNormalization(epsilon=1e-6)
        self.norm2 = layers.LayerNormalization(epsilon=1e-6)
        self.norm3 = layers.LayerNormalization(epsilon=1e-6)
        self.drop1 = layers.Dropout(dropout)
        self.drop2 = layers.Dropout(dropout)
        self.drop3 = layers.Dropout(dropout)

    def call(self, x, encoder_output, training=False, look_ahead_mask=None):
        attn1_out, _ = self.attn1(x, x, x, attention_mask=look_ahead_mask,
                                return_attention_scores=True, training=training)
        attn1_out = self.drop1(attn1_out, training=training)
        out1 = self.norm1(x + attn1_out)
        attn2_out, _ = self.attn2(out1, encoder_output, encoder_output,
                                return_attention_scores=True, training=training)
        attn2_out = self.drop2(attn2_out, training=training)
        out2 = self.norm2(out1 + attn2_out)
        ff_out = self.ff2(self.ff1(out2))
        ff_out = self.drop3(ff_out, training=training)
        return self.norm3(out2 + ff_out)

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"embed_dim": self.embed_dim, "num_heads": self.num_heads, "ff_dim": self.ff_dim})
        return cfg


class PositionalEmbedding(layers.Layer):
    def __init__(self, vocab_size, embed_dim, max_len, **kwargs):
        super().__init__(**kwargs)
        self.vocab_size = vocab_size
        self.embed_dim  = embed_dim
        self.max_len    = max_len
        self.token_emb  = layers.Embedding(vocab_size, embed_dim, mask_zero=False)
        self.pos_enc    = self._build_pos_enc(max_len, embed_dim)

    def _build_pos_enc(self, max_len, d):
        pos  = np.arange(max_len)[:, None]
        dims = np.arange(d)[None, :]
        ang  = pos / np.power(10000, (2 * (dims // 2)) / np.float32(d))
        ang[:, 0::2] = np.sin(ang[:, 0::2])
        ang[:, 1::2] = np.cos(ang[:, 1::2])
        return tf.cast(ang[None, :, :], tf.float32)

    def call(self, x):
        seq_len = tf.shape(x)[1]
        tok = self.token_emb(x) * tf.math.sqrt(tf.cast(self.embed_dim, tf.float32))
        return tok + self.pos_enc[:, :seq_len, :]

    def get_config(self):
        cfg = super().get_config()
        cfg.update({"vocab_size": self.vocab_size, "embed_dim": self.embed_dim, "max_len": self.max_len})
        return cfg


# ── Fixed causal mask (Keras 3 compatible) ────────────────────────────────────
def get_causal_mask(seq_len):
    return tf.linalg.band_part(tf.ones((seq_len, seq_len)), -1, 0)

def build_model4_efficientnet_transformer(vocab_size):
    img_inp  = Input(shape=(*IMG_SIZE, IMG_CHANNELS), name="image_input")
    cap_inp  = Input(shape=(MAX_SEQ_LEN - 1,), name="caption_input", dtype=tf.int32)
    cnn      = build_efficientnetb0_extractor(spatial=True)
    img_feat = cnn(img_inp)

    img_enc = layers.Dense(EMBED_DIM, activation="relu",
                        name="img_proj")(img_feat)                # ✅ activation="relu"
    img_enc = layers.LayerNormalization(epsilon=1e-6,
                        name="img_layernorm")(img_enc)            # ✅ added
    img_enc = layers.Dropout(DROPOUT_RATE, name="img_drop")(img_enc) # ✅ added

    dec_in  = PositionalEmbedding(vocab_size, EMBED_DIM, MAX_SEQ_LEN,
                        name="pos_embedding")(cap_inp)            # ✅ name fixed
    dec_in  = layers.Dropout(DROPOUT_RATE, name="cap_drop")(dec_in) # ✅ added

    mask    = get_causal_mask(MAX_SEQ_LEN - 1)

    x = dec_in
    for i in range(2):
        x = TransformerDecoderBlock(
            EMBED_DIM, NUM_HEADS, FF_DIM, DROPOUT_RATE,
            name=f"transformer_block_{i}"                            # ✅ name fixed
        )(x, img_enc, look_ahead_mask=mask)

    x      = layers.Dropout(DROPOUT_RATE, name="final_drop")(x)     # ✅ added
    logits = layers.Dense(vocab_size, activation="softmax",
                        name="output_softmax")(x)                  # ✅ name + activation fixed

    return Model([img_inp, cap_inp], logits,
                name="Model4_EfficientNetB0_Transformer")
print("✅ Model architecture defined")


✅ Model architecture defined


In [5]:
# ── EfficientNetB0 feature extractor (must match training notebook exactly) ───
def build_efficientnetb0_extractor(spatial: bool = False):
    """
    EfficientNetB0 feature extractor — identical to training notebook cell 18.

    spatial=True  → output (B, 49, 1280)  for Transformer cross-attention
    spatial=False → output (B, 1280)       for embedding/similarity search
    """
    base = EfficientNetB0(
        weights="imagenet",
        include_top=False,
        input_shape=(*IMG_SIZE, IMG_CHANNELS)
    )
    # Freeze all but last 20 layers (matches training exactly)
    for layer in base.layers[:-20]:
        layer.trainable = False
    for layer in base.layers[-20:]:
        layer.trainable = True

    inp          = base.input
    spatial_feat = base.get_layer("top_activation").output   # (B, 7, 7, 1280)

    if spatial:
        out = layers.Reshape((49, 1280), name="spatial_reshape")(spatial_feat)
        return Model(inp, out, name="EfficientNetB0_spatial")
    else:
        out = layers.GlobalAveragePooling2D(name="gap")(spatial_feat)
        return Model(inp, out, name="EfficientNetB0_vector")


print("✅ build_efficientnetb0_extractor() defined")
print("   spatial=True  → (B, 49, 1280)  for Transformer")
print("   spatial=False → (B, 1280)       for retrieval")

✅ build_efficientnetb0_extractor() defined
   spatial=True  → (B, 49, 1280)  for Transformer
   spatial=False → (B, 1280)       for retrieval


## Section 5 — Load Model & Tokenizer

In [7]:
# ── Load tokenizer ───────────────────────────────────────────────────────────
if TOKENIZER_PATH.exists():
    with open(TOKENIZER_PATH, "r") as f:
        tokenizer_json = json.load(f)
    tokenizer = tokenizer_from_json(tokenizer_json)
    vocab_size = len(tokenizer.word_index) + 1
    print(f"✅ Tokenizer loaded  —  vocab size: {vocab_size:,}")
else:
    raise FileNotFoundError(f"Tokenizer not found: {TOKENIZER_PATH}")

# ── Load model weights ────────────────────────────────────────────────────────
tf.keras.backend.clear_session()
captioning_model = build_model4_efficientnet_transformer(vocab_size=vocab_size)

if MODEL4_WEIGHTS.exists():
    captioning_model.load_weights(str(MODEL4_WEIGHTS))
    print(f"✅ Weights loaded from {MODEL4_WEIGHTS}")
else:
    print(f"⚠️  Weight file not found: {MODEL4_WEIGHTS}")
    print("    Model will produce random captions until weights are loaded.")

_d = np.zeros((1, *IMG_SIZE, IMG_CHANNELS), dtype=np.float32)
_c = np.zeros((1, MAX_SEQ_LEN - 1), dtype=np.int32)
_o = captioning_model([_d, _c], training=False)
print(f"✅ Forward pass OK — output shape: {_o.shape}")


✅ Tokenizer loaded  —  vocab size: 3,837
✅ Weights loaded from saved_models\model4_efficientnetb0_transformer_best.keras
✅ Forward pass OK — output shape: (1, 31, 3837)


## Section 6 — Image Utilities

In [8]:
def load_image_from_disk(product_id, img_dir=IMG_DIR, img_size=IMG_SIZE):
    path = Path(img_dir) / f"{product_id}.jpg"
    try:
        if path.exists():
            with Image.open(path) as img:
                img = img.convert("RGB").resize(img_size, Image.BILINEAR)
                return np.array(img, dtype=np.float32) / 255.0
    except Exception:
        pass
    return np.ones((*img_size, 3), dtype=np.float32) * 0.5

def load_image_from_url(url, img_size=IMG_SIZE):
    try:
        resp = requests.get(url, timeout=10)
        resp.raise_for_status()
        img = Image.open(BytesIO(resp.content)).convert("RGB")
        img = img.resize(img_size, Image.BILINEAR)
        return np.array(img, dtype=np.float32) / 255.0
    except Exception as e:
        print(f"  ⚠️  URL load failed ({e}) — using placeholder")
        return np.ones((*img_size, 3), dtype=np.float32) * 0.5

def load_image_from_path(file_path, img_size=IMG_SIZE):
    try:
        with Image.open(file_path) as img:
            img = img.convert("RGB").resize(img_size, Image.BILINEAR)
            return np.array(img, dtype=np.float32) / 255.0
    except Exception as e:
        print(f"  ⚠️  File load failed ({e}) — using placeholder")
        return np.ones((*img_size, 3), dtype=np.float32) * 0.5

def preprocess_for_efficientnet(img_01):
    img_255 = img_01 * 255.0
    return np.array(
        tf.keras.applications.efficientnet.preprocess_input(img_255),
        dtype=np.float32
    )

print("✅ Image utilities ready")


✅ Image utilities ready


## Section 7 — Caption Generation

In [9]:
def generate_caption(model, img_01, max_len=MAX_SEQ_LEN, verbose=False):
    """Auto-regressive greedy decoding."""
    img_pre = preprocess_for_efficientnet(img_01)
    img_pre = np.expand_dims(img_pre, 0).astype(np.float32)

    start_id = tokenizer.word_index.get(START_TOKEN, 1)
    end_id   = tokenizer.word_index.get(END_TOKEN,   2)

    token_ids = [start_id]
    idx2word  = {v: k for k, v in tokenizer.word_index.items()}

    for step in range(max_len - 1):
        seq_pad = pad_sequences([token_ids], maxlen=MAX_SEQ_LEN - 1,
                                padding="post", truncating="post")
        logits  = model.predict([img_pre, seq_pad], verbose=0)
        next_id = int(np.argmax(logits[0, len(token_ids) - 1]))
        if verbose:
            print(f"  step {step:2d}: {next_id} → {idx2word.get(next_id,'<unk>')}")
        if next_id == end_id:
            break
        token_ids.append(next_id)

    skip  = {START_TOKEN, END_TOKEN, PAD_TOKEN, UNK_TOKEN, ""}
    words = [idx2word.get(i,"") for i in token_ids[1:]]
    words = [w for w in words if w and w not in skip]
    return " ".join(words)

print("✅ generate_caption() ready")


✅ generate_caption() ready


## Section 8 — Embedding Extractor

In [10]:
# Re-use fine-tuned EfficientNetB0 weights from captioning model
effnet_submodel = None
for layer in captioning_model.layers:
    if "efficientnetb0" in layer.name.lower():
        effnet_submodel = layer
        break

inp = Input(shape=(*IMG_SIZE, IMG_CHANNELS), name="image_input")
if effnet_submodel is not None:
    spatial_out = effnet_submodel(inp)                         # (B, 49, 1280)
    reshaped    = layers.Reshape((7, 7, 1280))(spatial_out)
    pooled      = layers.GlobalAveragePooling2D()(reshaped)    # (B, 1280)
    print(f"✅ Using fine-tuned EfficientNetB0: {effnet_submodel.name}")
else:
    base = EfficientNetB0(weights="imagenet", include_top=False,
                        input_shape=(*IMG_SIZE, IMG_CHANNELS))
    base.trainable = False
    feat_map = base.get_layer("top_activation").output
    pooled   = layers.GlobalAveragePooling2D()(feat_map)
    inp      = base.input
    print("⚠️  Using fresh ImageNet weights (fine-tuned model not found)")

embedding_extractor = Model(inp, pooled, name="effnet_embedding_extractor")
embedding_extractor.trainable = False

def extract_embedding(img_01):
    img_pre = preprocess_for_efficientnet(img_01)
    img_pre = np.expand_dims(img_pre, 0).astype(np.float32)
    emb     = embedding_extractor.predict(img_pre, verbose=0)[0]
    return emb / (np.linalg.norm(emb) + 1e-8)

print("✅ extract_embedding() ready")


✅ Using fine-tuned EfficientNetB0: EfficientNetB0_spatial
✅ extract_embedding() ready


## Section 9 — Load Cached Embeddings

In [11]:
if EMBEDDINGS_PATH.exists() and PRODUCT_IDS_PATH.exists():
    product_embeddings = np.load(EMBEDDINGS_PATH)
    product_ids        = np.load(PRODUCT_IDS_PATH)
    print(f"✅ Cached embeddings loaded")
    print(f"   Embeddings : {product_embeddings.shape}")
    print(f"   Product IDs: {product_ids.shape}")
else:
    raise FileNotFoundError(
        f"Embeddings not found at {EMBEDDINGS_DIR}/.\n"
        "Please run Section 9 in retrieval_system.ipynb first to generate and cache embeddings."
    )

def cosine_similarity_search(query_emb, db_embeddings, db_product_ids, top_k=TOP_K):
    q    = query_emb.reshape(1, -1).astype(np.float32)
    sims = cosine_similarity(q, db_embeddings)[0]
    top_idx = np.argsort(sims)[::-1][:top_k]
    return pd.DataFrame({
        "product_id"      : db_product_ids[top_idx],
        "similarity_score": sims[top_idx]
    }).reset_index(drop=True)

print("✅ cosine_similarity_search() ready")


✅ Cached embeddings loaded
   Embeddings : (5281, 1280)
   Product IDs: (5281,)
✅ cosine_similarity_search() ready


## Section 10 — 💬 Chatbot Interface

Send an image (local path or URL) and the chatbot will:
1. Display your uploaded image
2. Generate an AI caption using EfficientNetB0 + Transformer
3. Show the **Top-5 most visually similar** products with similarity scores

**Run the cell below and use the widget.**


In [ ]:
# ════════════════════════════════════════════════════════════════════════
# Section 10 — Intelligent Fashion Chatbot (FINAL FIX)
# ════════════════════════════════════════════════════════════════════════

import io, base64
from sklearn.feature_extraction.text import TfidfVectorizer
from IPython.display import display, HTML
import ipywidgets as widgets
import matplotlib.pyplot as plt

# ────────────────────────────────────────────────────────────────────────
# 10.1  TF-IDF index
# ────────────────────────────────────────────────────────────────────────
df["_search_text"] = (
    df["title_clean"].fillna("") + " " +
    df["category_clean"].fillna("") + " " +
    df["primary_color"].fillna("") + " " +
    df["gender_clean"].fillna("") + " " +
    df["price_bucket"].fillna("") + " " +
    df["caption"].fillna("")
)
_tfidf     = TfidfVectorizer(max_features=8000, ngram_range=(1,3), sublinear_tf=True)
_tfidf_mat = _tfidf.fit_transform(df["_search_text"].tolist())
print("✅ TF-IDF index built")

# ────────────────────────────────────────────────────────────────────────
# 10.2  Input-type detector
# ────────────────────────────────────────────────────────────────────────
def _detect_input_type(text: str) -> str:
    t = text.strip()
    if t.lower().startswith(("http://","https://")):
        return "url"
    if (any(t.lower().endswith(e) for e in (".jpg",".jpeg",".png",".bmp",".webp"))
            or "\\" in t or (len(t) > 6 and "/" in t and " " not in t)):
        return "path"
    return "text"

# ────────────────────────────────────────────────────────────────────────
# 10.3  Search pipelines
# ────────────────────────────────────────────────────────────────────────
_COLS = ["product_id","similarity_score","title_clean","category_clean",
        "gender_clean","price_egp","price_bucket","primary_color",
        "brand","product_url","img_url"]

def _search_text(query: str) -> pd.DataFrame:
    vec  = _tfidf.transform([query])
    sims = cosine_similarity(vec, _tfidf_mat)[0]
    idx  = np.argsort(sims)[::-1][:TOP_K]
    res  = df.iloc[idx].copy()
    res["similarity_score"] = sims[idx]
    return res[[c for c in _COLS if c in res.columns]].reset_index(drop=True)

def _search_image(src: str):
    img = (load_image_from_url(src)
        if src.lower().startswith(("http://","https://"))
        else load_image_from_path(src))
    cap = generate_caption(captioning_model, img)
    emb = extract_embedding(img)
    sdf = cosine_similarity_search(emb, product_embeddings, product_ids, TOP_K)
    rdf = sdf.merge(df, on="product_id", how="left")
    return img, cap, rdf[[c for c in _COLS if c in rdf.columns]].reset_index(drop=True)

# ────────────────────────────────────────────────────────────────────────
# 10.4  Image helpers
# ────────────────────────────────────────────────────────────────────────
def _img_to_b64(arr: np.ndarray, size: int = 90) -> str:
    fig, ax = plt.subplots(figsize=(1,1))
    fig.patch.set_facecolor("#0f172a")
    ax.imshow(arr); ax.axis("off")
    buf = io.BytesIO()
    fig.savefig(buf, format="png", bbox_inches="tight",
                facecolor="#0f172a", dpi=size)
    plt.close(fig); buf.seek(0)
    return base64.b64encode(buf.read()).decode()

def _product_b64(pid, img_url: str) -> str:
    path = Path(IMG_DIR) / f"{pid}.jpg"
    try:
        arr = (load_image_from_disk(pid) if path.exists()
            else load_image_from_url(str(img_url))
            if str(img_url) not in ("nan","","None") else None)
        if arr is not None:
            return _img_to_b64(arr)
    except Exception:
        pass
    return _img_to_b64(np.full((*IMG_SIZE, 3), 0.15, dtype=np.float32))

# ────────────────────────────────────────────────────────────────────────
# 10.5  HTML card builders
# ────────────────────────────────────────────────────────────────────────
_SC = ["#22c55e","#84cc16","#eab308","#f97316","#ef4444"]
_RK = ["🥇","🥈","🥉","4️⃣","5️⃣"]

def _caption_banner(query_img, caption: str) -> str:
    qb = _img_to_b64(query_img, size=100)
    return f"""
    <div style="display:flex;align-items:center;gap:12px;
                background:linear-gradient(135deg,#1e1b4b,#1e293b);
                border-radius:12px;padding:12px 16px;margin-bottom:12px;
                border:1px solid #312e81;">
    <img src="data:image/png;base64,{qb}"
        style="width:76px;height:76px;border-radius:10px;
                object-fit:cover;flex-shrink:0;border:2px solid #7c3aed;"/>
    <div>
        <div style="color:#a78bfa;font-weight:700;font-size:11px;
                    margin-bottom:5px;letter-spacing:1px;">
        🤖 AI-Generated Caption</div>
        <div style="color:#e2e8f0;font-style:italic;font-size:13px;
                    line-height:1.6;">"{caption}"</div>
    </div>
    </div>"""

def _result_card(i, row) -> str:
    score = float(row.get("similarity_score", 0))
    title = str(row.get("title_clean",    "N/A"))
    cat   = str(row.get("category_clean", "N/A"))
    gen   = str(row.get("gender_clean",   "N/A"))
    col   = str(row.get("primary_color",  "N/A"))
    price = str(row.get("price_egp",      "N/A"))
    bkt   = str(row.get("price_bucket",   ""))
    brand = str(row.get("brand",          "N/A"))
    url   = str(row.get("product_url",    ""))
    pid   = row.get("product_id")
    iurl  = str(row.get("img_url",        ""))
    sc    = _SC[min(i, 4)]
    rk    = _RK[min(i, 4)]
    ib    = _product_b64(pid, iurl)
    bar_w = int(score * 100)

    link = ""
    if url not in ("nan","","None"):
        link = (f'<a href="{url}" target="_blank" '
                f'style="display:inline-block;margin-top:6px;'
                f'background:#1e3a5f;color:#60a5fa;font-size:11px;'
                f'text-decoration:none;padding:4px 10px;'
                f'border-radius:6px;border:1px solid #2563eb44;">'
                f'🛒 View on Jumia</a>')

    return f"""
    <div style="display:flex;align-items:flex-start;gap:12px;
                background:linear-gradient(135deg,#0f172a,#0d1829);
                border:1px solid #1e293b;border-left:4px solid {sc};
                border-radius:12px;padding:12px;margin-bottom:10px;
                box-shadow:0 4px 15px #0004;">
    <div style="position:relative;flex-shrink:0;">
        <img src="data:image/png;base64,{ib}"
            style="width:88px;height:88px;border-radius:10px;
                    object-fit:cover;border:2px solid {sc}55;"/>
        <div style="position:absolute;top:-6px;left:-6px;
                    background:{sc};color:#000;font-size:11px;
                    font-weight:800;border-radius:50%;
                    width:22px;height:22px;
                    display:flex;align-items:center;justify-content:center;">
        {i+1}</div>
    </div>
    <div style="flex:1;min-width:0;">
        <div style="display:flex;justify-content:space-between;
                    align-items:flex-start;margin-bottom:6px;">
        <span style="color:white;font-size:13px;font-weight:700;
                    line-height:1.4;padding-right:8px;">
            {rk} {title[:60]}{"…" if len(title)>60 else ""}</span>
        <span style="background:{sc}22;color:{sc};flex-shrink:0;
                    border:1px solid {sc}66;border-radius:20px;
                    padding:3px 10px;font-size:12px;font-weight:800;">
            {score:.1%}</span>
        </div>
        <div style="background:#1e293b;border-radius:4px;
                    height:3px;margin-bottom:8px;overflow:hidden;">
        <div style="width:{bar_w}%;height:100%;
                    background:linear-gradient(90deg,{sc}88,{sc});
                    border-radius:4px;"></div>
        </div>
        <div style="display:flex;flex-wrap:wrap;gap:5px;margin-bottom:6px;">
        <span style="background:#1e293b;color:#94a3b8;
                    padding:2px 8px;border-radius:20px;font-size:10px;">
            📦 {cat}</span>
        <span style="background:#1e293b;color:#94a3b8;
                    padding:2px 8px;border-radius:20px;font-size:10px;">
            👤 {gen}</span>
        <span style="background:#1e293b;color:#94a3b8;
                    padding:2px 8px;border-radius:20px;font-size:10px;">
            🎨 {col}</span>
        <span style="background:#1e293b;color:#94a3b8;
                    padding:2px 8px;border-radius:20px;font-size:10px;">
            🏷️ {brand}</span>
        </div>
        <div style="color:#fbbf24;font-size:13px;font-weight:700;">
        💰 EGP {price}
        <span style="color:#475569;font-size:11px;font-weight:400;">
            ({bkt})</span>
        </div>
        {link}
    </div>
    </div>"""

def _build_bot_html(results_df, query_img=None, caption=None,
                    input_type="text") -> str:
    parts = []
    badge_map = {
        "text": ("💬 Text Search",  "#6366f1"),
        "url":  ("🌐 URL Search",   "#0ea5e9"),
        "path": ("📂 Image Search", "#8b5cf6"),
    }
    badge, bc = badge_map.get(input_type, ("🔍 Search","#6366f1"))
    parts.append(
        f'<div style="display:inline-block;background:{bc}22;color:{bc};'
        f'border:1px solid {bc}55;border-radius:20px;padding:3px 12px;'
        f'font-size:10px;font-weight:700;margin-bottom:10px;">{badge}</div>')
    if caption and query_img is not None:
        parts.append(_caption_banner(query_img, caption))
    parts.append(
        f'<div style="color:#475569;font-size:10px;font-weight:700;'
        f'letter-spacing:1.5px;margin-bottom:10px;text-transform:uppercase;">'
        f'🔍 Top {len(results_df)} Similar Products</div>')
    for i, (_, row) in enumerate(results_df.iterrows()):
        parts.append(_result_card(i, row))
    return "".join(parts)

# ────────────────────────────────────────────────────────────────────────
# 10.6  Chat bubble renderers
# ────────────────────────────────────────────────────────────────────────
_chat_log = []

def _user_bubble(text: str) -> str:
    return f"""
    <div style="display:flex;justify-content:flex-end;
                margin:10px 0 10px 40px;">
    <div style="background:linear-gradient(135deg,#4f46e5,#7c3aed);
                color:white;padding:10px 16px;
                border-radius:18px 18px 4px 18px;
                font-size:13px;word-break:break-word;
                box-shadow:0 4px 15px #7c3aed33;
                width:100%;">        <!-- ✅ added this -->
        <span style="opacity:.6;font-size:10px;display:block;
                    margin-bottom:3px;">You</span>
        {text}
    </div>
    </div>"""

def _bot_bubble(html: str) -> str:
    return f"""
    <div style="display:flex;justify-content:flex-start;
                margin:10px 0 10px 0;">     <!-- ✅ same vertical margin -->
    <div style="background:#0f172a;border:1px solid #1e293b;
                color:white;padding:14px 16px;
                border-radius:4px 18px 18px 18px;
                font-size:12px;line-height:1.6;width:100%;
                box-shadow:0 4px 15px #0006;">
        <span style="color:#475569;font-size:10px;display:block;
                    margin-bottom:6px;">🤖 Fashion AI</span>
        {html}
    </div>
    </div>"""

def _thinking_html() -> str:
    return """
    <div style="display:flex;justify-content:flex-start;margin:10px 0;">
    <div style="background:#0f172a;border:1px solid #1e293b;
                color:#64748b;padding:14px 20px;
                border-radius:4px 18px 18px 18px;font-size:13px;">
        ⏳ Searching through 5,281 products…
    </div>
    </div>"""

def _welcome_html() -> str:
    return """
    <div style="text-align:center;padding:50px 20px 30px;">
    <div style="font-size:48px;margin-bottom:14px;">🛍️</div>
    <div style="color:#a78bfa;font-size:18px;font-weight:800;
                margin-bottom:8px;">Fashion AI Assistant</div>
    <div style="color:#334155;font-size:12px;margin-bottom:20px;">
        Powered by EfficientNetB0 + Transformer · 5,281 Jumia Egypt products
    </div>
    <div style="display:flex;flex-direction:column;gap:8px;align-items:center;">
        <div style="background:#1e293b;color:#94a3b8;padding:8px 20px;
                    border-radius:20px;font-size:12px;border:1px solid #334155;">
        💬  "red sneakers for men under 500 egp"</div>
        <div style="background:#1e293b;color:#94a3b8;padding:8px 20px;
                    border-radius:20px;font-size:12px;border:1px solid #334155;">
        🌐  https://example.com/product-image.jpg</div>
        <div style="background:#1e293b;color:#94a3b8;padding:8px 20px;
                    border-radius:20px;font-size:12px;border:1px solid #334155;">
        📂  /path/to/local/image.jpg</div>
    </div>
    </div>"""

# ────────────────────────────────────────────────────────────────────────
# 10.7  THE KEY FIX — single HTML widget, scroll div is INSIDE the HTML
# ────────────────────────────────────────────────────────────────────────

def _build_full_ui_html(messages_html: str, status_text: str = "✨ Ready.") -> str:
    """
    Renders the ENTIRE chatbot as one HTML string.
    The scroll container is a plain <div> with overflow-y:auto inside the HTML.
    The input bar is OUTSIDE that div — it never scrolls.
    Real input is handled by ipywidgets below this HTML block.
    """
    return f"""
    <div style="
        max-width:99% ; margin:0 auto;
        border:1px solid #1e293b; border-radius:14px;
        overflow:hidden; font-family:'Segoe UI',sans-serif;
        background:#020617;">

    <!-- HEADER — never scrolls -->
    <div style="
        background:linear-gradient(135deg,#0f0c29,#302b63,#1a1a3e);
        padding:14px 20px; border-bottom:2px solid #312e81;
        flex-shrink:0;">
        <div style="display:flex;align-items:center;
                    justify-content:space-between;">
        <div style="display:flex;align-items:center;gap:12px;">
            <div style="background:linear-gradient(135deg,#7c3aed,#4f46e5);
                        width:40px;height:40px;border-radius:12px;
                        display:flex;align-items:center;justify-content:center;
                        font-size:20px;">🛍️</div>
            <div>
            <div style="color:#e2e8f0;font-size:15px;font-weight:800;">
                Fashion AI Chatbot</div>
            <div style="color:#64748b;font-size:11px;margin-top:2px;">
                EfficientNetB0 + Transformer · Jumia Egypt</div>
            </div>
        </div>
        <div style="display:flex;gap:6px;">
            <div style="background:#22c55e22;color:#22c55e;
                        border:1px solid #22c55e44;border-radius:20px;
                        padding:3px 12px;font-size:10px;font-weight:700;">
            ● Live</div>
            <div style="background:#6366f122;color:#6366f1;
                        border:1px solid #6366f144;border-radius:20px;
                        padding:3px 12px;font-size:10px;">
            5,281 products</div>
        </div>
        </div>
    </div>

    <!-- MESSAGES — THIS div scrolls, nothing else -->
    <div id="chat-scroll-area" style="
        height:680px;
        overflow-y:auto;
        overflow-x:hidden;
        padding:14px 16px;
        background:#020617;">
        {messages_html}
        <div id="chat-anchor"></div>
    </div>

    <!-- STATUS BAR — never scrolls -->
    <div style="
        background:#0a0f1e;
        border-top:1px solid #1e293b;
        padding:5px 16px 6px;
        font-size:11px; color:#475569;">
        {status_text}
    </div>

    </div>

    <script>
      // Auto-scroll the inner div to bottom
    var el = document.getElementById('chat-scroll-area');
    if(el) el.scrollTop = el.scrollHeight;
    </script>
    """

# ── The HTML display widget (shows header + scrollable chat + status) ─────
_chat_display = widgets.HTML(value=_build_full_ui_html(_welcome_html()))

# ── Input row — completely separate widget, ALWAYS below the HTML ──────────
_inp = widgets.Text(
    placeholder='💬 Describe a product  |  🌐 Image URL  |  📂 Local path',
    layout=widgets.Layout(flex="1", height="42px")
)
_btn_send = widgets.Button(
    description="Send ➤",
    button_style="primary",
    layout=widgets.Layout(width="90px", height="42px")
)
_btn_clear = widgets.Button(
    description="🗑️ Clear",
    button_style="warning",
    layout=widgets.Layout(width="84px", height="42px")
)

_input_row = widgets.HBox(
    [_inp, _btn_send, _btn_clear],
    layout=widgets.Layout(
        padding="10px 14px",
        gap="8px",
        background_color="#0f172a",
        border="1px solid #1e293b",
        border_top="2px solid #312e81",
        border_radius="0 0 14px 14px",
    )
)

# ────────────────────────────────────────────────────────────────────────
# 10.8  Chat state helpers
# ────────────────────────────────────────────────────────────────────────
def _messages_html(extra: str = "") -> str:
    if not _chat_log:
        return _welcome_html()
    parts = [
        _user_bubble(c) if r == "user" else _bot_bubble(c)
        for r, c in _chat_log
    ]
    return "".join(parts) + extra

def _refresh(status: str = "✨ Ready.", extra: str = ""):
    _chat_display.value = _build_full_ui_html(
        _messages_html(extra), status_text=status)

# ────────────────────────────────────────────────────────────────────────
# 10.9  Event handlers
# ────────────────────────────────────────────────────────────────────────
def _on_send(_):
    src = _inp.value.strip()
    if not src:
        _refresh("⚠️ Please enter something.")
        return

    _inp.value = ""
    _chat_log.append(("user", src))
    _refresh("⏳ Searching…", extra=_thinking_html())

    try:
        itype = _detect_input_type(src)
        if itype in ("url", "path"):
            _refresh("🖼️ Loading image & generating caption…",
                    extra=_thinking_html())
            qi, cap, rdf = _search_image(src)
            bot_html = _build_bot_html(rdf, qi, cap, input_type=itype)
        else:
            rdf      = _search_text(src)
            bot_html = _build_bot_html(rdf, input_type="text")

        _chat_log.append(("bot", bot_html))
        _refresh(f"✅ Found {len(rdf)} matches.")

    except Exception as e:
        _chat_log.append(("bot",
            f"<span style='color:#f87171;'>❌ {e}</span>"))
        _refresh(f"❌ Error: {e}")

def _on_clear(_):
    _chat_log.clear()
    _refresh("🗑️ Cleared — ready.")

_inp.on_submit(_on_send)
_btn_send.on_click(_on_send)
_btn_clear.on_click(_on_clear)

# ────────────────────────────────────────────────────────────────────────
# 10.10  Final assembly
#  _chat_display  → header + scrollable chat div + status  (one HTML widget)
#  _input_row     → always below, never scrolls
# ────────────────────────────────────────────────────────────────────────
_ui = widgets.VBox(
    [_chat_display, _input_row],
    layout=widgets.Layout(
        width="99%",
        margin="10px auto"
    )
)

display(_ui)

✅ TF-IDF index built
